# ModelRouter Pipeline (AI-Driven Flow with Human-in-the-Loop)
In this automated workflow, the dataset is profiled and sent to an AI router. 
The AI analyzes the data and recommends the best machine learning method from the shared registry, 
providing its reasoning. The notebook halts to wait for your confirmation before executing the suggested method.

In [23]:
import sys
import os
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

sys.path.append(os.path.abspath('..'))

from model_router.src.profiler import profile_dataset
from model_router.src.router import recommend_method
from model_router.src.report import generate_model_report
from shared.registry import get_method

from IPython.display import display, clear_output
from shared.llm_client import OpenRouterClient

from shared.visualizations import display_execution_plots


In [24]:
data_dir = '../data'
os.makedirs(data_dir, exist_ok=True)
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]

if not csv_files:
    print('No CSV files found in ../data. A dummy dataset will be used.')
else:
    dataset_dropdown = widgets.Dropdown(options=csv_files, description='Dataset:')
    display(dataset_dropdown)

Dropdown(description='Dataset:', options=('AB_NYC_2019.csv', 'creditcard_csv.csv', 'Mall_Customers.csv', 'Onli…

In [25]:
if 'dataset_dropdown' in locals() and csv_files:
    selected_file = dataset_dropdown.value
    df = pd.read_csv(os.path.join(data_dir, selected_file))
    print(f'Loaded {selected_file} with shape {df.shape}')
else:
    print("Please select a dataset.")

Loaded WA_Fn-UseC_-HR-Employee-Attrition.csv with shape (1470, 35)


In [26]:
print("Asking AI to identify the target column...")
client = OpenRouterClient()
columns_list = list(df.columns)
dataset_name = selected_file if 'selected_file' in locals() else "Unknown Dataset"

target_prompt = f"""
I have a dataset named '{dataset_name}' with the following columns:
{columns_list}

Which of these columns is the most likely 'target' variable for a machine learning model? 
If there is no obvious target variable (e.g. it is a clustering dataset like Mall Customers), reply with "None".
You must output ONLY the exact column name or "None", with no other text, markdown, quotes, or punctuation.
"""

ai_response = client.generate(target_prompt, system_prompt="You are an expert data scientist.").strip()

# Clean up any potential markdown formatting from the AI
ai_response = ai_response.replace('`', '').replace('"', '').replace("'", "").strip()

if ai_response in df.columns:
    target_col = ai_response
    print(f"🤖 AI intelligently identified target column: '{target_col}'")
else:
    target_col = None
    print("🤖 AI determined this is likely an unsupervised dataset (No target).")

Asking AI to identify the target column...
🤖 AI intelligently identified target column: 'Attrition'


In [27]:
# 1. Profile Dataset
profile = profile_dataset(df, target_col=target_col)
print("Dataset Profile Extraction Complete.")

Dataset Profile Extraction Complete.


In [28]:
# 2. Request AI Routing Recommendation
dataset_description = "A tabular dataset containing numeric and categorical features."
print("Asking AI Router for recommendation...")
recommendation = recommend_method(dataset_description, profile)

print("\n--- AI Recommendation ---")
print(f"Method: {recommendation.get('recommended_method')}")
print(f"Problem Type: {recommendation.get('problem_type')}")
print(f"Reasoning: {recommendation.get('reasoning')}")

Asking AI Router for recommendation...
Error with model tencent/hy3:free: 400 Client Error: Bad Request for url: https://openrouter.ai/api/v1/chat/completions

--- AI Recommendation ---
Method: random_forest_classifier
Problem Type: classification
Reasoning: The dataset is a binary classification problem with a mix of numeric and categorical features. The target variable 'Attrition' has two classes ('No' and 'Yes'), and the problem type is explicitly stated as classification. Among the available methods, only 'random_forest_classifier' is a classification algorithm. Random Forest is well-suited for this dataset because: (1) it handles mixed data types (numeric/categorical) natively after encoding, (2) it is robust to class imbalance (83.87% vs. 16.12% class distribution), and (3) it can capture non-linear relationships without requiring feature scaling. Linear regression and k-means clustering are unsuitable for this classification task.


In [ ]:
# 3. Conversational Human-in-the-Loop Confirmation
confirm_button = widgets.Button(description='Confirm & Execute', button_style='success')
cancel_button = widgets.Button(description='Reject & Question AI', button_style='danger')
output_area = widgets.Output()

CONFIRMED_TO_RUN = False
current_recommendation = recommendation.copy()

def on_confirm(b):
    global CONFIRMED_TO_RUN
    CONFIRMED_TO_RUN = True
    with output_area:
        output_area.clear_output()
        print('✅ Recommendation CONFIRMED! Please proceed to run the next cell to execute the model.')

def on_reject(b):
    global CONFIRMED_TO_RUN
    CONFIRMED_TO_RUN = False
    with output_area:
        output_area.clear_output()
        
        question_input = widgets.Textarea(
            placeholder="Why didn't you choose X? / The data is actually highly imbalanced...",
            description="Your Question:",
            layout=widgets.Layout(width='80%', height='100px')
        )
        send_button = widgets.Button(description='Send to AI', button_style='primary')
        chat_output = widgets.Output()
        
        display(widgets.VBox([question_input, send_button, chat_output]))
        
        def on_send(btn):
            with chat_output:
                chat_output.clear_output()
                print("Thinking...")
                client = OpenRouterClient()
                prompt = f"""
                You previously recommended '{current_recommendation.get('recommended_method')}' for this dataset.
                Your Reasoning: {current_recommendation.get('reasoning')}
                
                The user rejected this and asked: "{question_input.value}"
                
                Respond to the user directly. Defend your choice or update it. Be concise.
                """
                response = client.generate(prompt, system_prompt="You are an expert Data Scientist.")
                chat_output.clear_output()
                print(f"🤖 AI: {response}")
                
        send_button.on_click(on_send)

confirm_button.on_click(on_confirm)
cancel_button.on_click(on_reject)
display(widgets.HBox([confirm_button, cancel_button]), output_area)

Output()

In [31]:
# 4. Execute Method (With Inline Plots and AI Summary)
import os
from shared.registry import get_method
from model_router.src.report import generate_model_report
from shared.visualizations import display_execution_plots
from shared.llm_client import OpenRouterClient

if not globals().get('CONFIRMED_TO_RUN'):
    print('Execution halted. Please confirm the LLM recommendation in the previous cell.')
else:
    method_name = current_recommendation.get('recommended_method')
    problem_type = current_recommendation.get('problem_type')
    print(f"Executing {method_name}...")
    
    try:
        func = get_method(method_name)
        X = df.drop(columns=[target_col]) if target_col else df
        y = df[target_col] if target_col else None
        
        results = func(X, y)
        
        print("\n--- Visualizing Results ---")
        display_execution_plots(problem_type, results)
        
        print("\n--- Generating Inline AI Summary ---")
        client = OpenRouterClient()
        summary_prompt = f"""
        Summarize these model metrics for a business stakeholder: {results.get('metrics')}
        
        Formatting requirements:
        - Write exactly 2-3 short, highly readable paragraphs.
        - Do NOT use a single dense block of text.
        - Do NOT use emojis.
        - Focus on the business impact of the metrics (e.g., if recall is low, what does that mean for catching fraud?).
        """
        inline_summary = client.generate(summary_prompt, system_prompt="You are an expert Data Scientist.")
        print(f"AI Summary:\n{inline_summary}\n")
        
        output_path = "reports/model_report.html"
        generate_model_report(current_recommendation, results, output_path=output_path)
    except Exception as e:
        print(f"Execution Failed: {e}")

Executing random_forest_classifier...

--- Visualizing Results ---
No predictions available to plot.

--- Generating Inline AI Summary ---
AI Summary:
The model shows an overall accuracy of 87.8%, which looks strong at a glance. However, this is driven by excellent performance on the "No" class (255 cases), where the model correctly identifies 98.8% of actual negatives. Because "No" dominates the data, the high accuracy masks a serious weakness in detecting the rarer "Yes" cases.

For the "Yes" class (only 39 cases), the model suffers from very low recall at 15.4%. In business terms, if "Yes" represents fraud, churn, or another critical event, the model would miss over 84% of those occurrences. Although its precision for "Yes" is 66.7% (most flags are correct), the failure to catch the majority of true positives limits its reliability for proactive intervention.

Model report generated at reports/model_report.html
